In [ ]:
# these are the libraries that you will need throughout the assignment
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline


### loading the merged dataset

In [ ]:
final_df = pd.DataFrame() # change this
final_df = pd.read_csv("final_df.csv", low_memory=False)

#header=1 bc we omit row 0, column names start at row=1

final_df.shape

In [ ]:
print('Data Show Describe\n')
final_df.describe()

## Checking missing values for demographics

In [ ]:
#checking demographic columns, disease timing/backgound (ageonset,
#agediag, age_DATSCAN)

demo_cols = [
    "age",
    "SEX",
    "race", "COHORT", "CONCOHORT", "subgroup", "ageonset", "agediag",
    "age_at_visit", "educ", "EDUCYRS",

]

missing_values_demo = final_df[demo_cols].isna().sum()

print(missing_values_demo)


missing_pct = final_df[demo_cols].isna().mean() * 100

plt.figure()
missing_pct.plot(kind="bar", color = ['pink'])
plt.ylabel("Percentage missing (%)")
plt.title("Percentage of missing values demo-variables")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
(final_df
 .groupby("CONCOHORT")
 .agg(
     age_mean=("age", "mean"),
     age_sd=("age", "std"),
     ageonset_mean=("ageonset", "mean"),
     ageonset_sd=("ageonset", "std"),
     n=("age", "count")
 ))


## Replacing missing concohort values with corresponding cohort values

In [ ]:
##replaces missing concohort values with corresponding cohort values
final_df["CONCOHORT"] = final_df["CONCOHORT"].fillna(final_df["COHORT"])

#### Dropping rows with missing values in agediag for PD patients
#### Also dropping rows WITH values in agediag for Prodromal cases

In [ ]:
##Dropping rows from PD patients with missing values for "agediag"
final_df = final_df.drop(
    final_df[
    (final_df["CONCOHORT"] == 1) & (final_df["agediag"].isna())].index ) 
final_df.shape

In [ ]:
##checking prodromal (CNCOHORT 3)cases with values for "agediag"
final_df["agediag"] = pd.to_numeric(
    final_df["agediag"].astype("string").str.strip().str.replace(",", ".", regex=False),
    errors="coerce"
)

# Prodromal cohort = 4, and KEEP those with agediag present
prod_with_agediag = final_df[
    final_df["CONCOHORT"].eq(4) & final_df["agediag"].notna()
].copy()

print("Prodromal rows with agediag:", len(prod_with_agediag))
print("Unique Prodromal IDs with agediag:", prod_with_agediag["PATNO"].nunique())

prodromal_ids_with_agediag = (prod_with_agediag["PATNO"]
                              .dropna()
                              .drop_duplicates()
                              .sort_values()
                              .tolist())

prodromal_ids_with_agediag

In [ ]:
final_df.shape

In [ ]:
##REMOVING rows of prodromal patients with values for "agediag"
final_df=final_df[~final_df["PATNO"].isin(prodromal_ids_with_agediag)]
final_df.shape 


In [ ]:
final_df["CONCOHORT"].isna().sum()

#### Continuing with all concohorts except 3 (i.e. SWEDD/PD)

In [ ]:
df_cohorts = final_df[final_df["CONCOHORT"] !=3]

In [ ]:
df_cohorts["CONCOHORT"].value_counts()

In [ ]:
df_cohorts["subgroup"].value_counts()

In [ ]:
pd.crosstab(df_cohorts["subgroup"], df_cohorts["CONCOHORT"])

In [ ]:
label_map = {
    1: "PD",
    2: "HC",
    4: "Prodromal PD"
}

# Count relevant cohorts and enforce display order
counts = (
    final_df
    .loc[final_df["CONCOHORT"].isin(label_map.keys()), "CONCOHORT"]
    .value_counts()
    .reindex([1, 4, 2])
)

# Map numeric cohort codes to labels
x_labels = counts.index.map(label_map)

plt.figure()
plt.bar(x_labels, counts.values)
plt.xlabel("Cohort")
plt.ylabel("Number of patients")
plt.title("Number of Patients per Cohort")
plt.show()


In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal PD"}
sex_map    = {0: "Female", 1: "Male"}

# Build a contingency table (counts) by cohort x sex
ct = (
    final_df
    .loc[final_df["CONCOHORT"].isin(cohort_map.keys()), ["CONCOHORT", "SEX"]]
    .assign(
        CONCOHORT_lbl=lambda d: d["CONCOHORT"].map(cohort_map),
        SEX_lbl=lambda d: d["SEX"].map(sex_map),
    )
    .groupby(["CONCOHORT_lbl", "SEX_lbl"])
    .size()
    .unstack(fill_value=0)
    .reindex(["PD", "Prodromal PD", "HC"])  # desired x-axis order
)

ax = ct.plot(kind="bar", stacked=True, color=["pink", "lightblue"])
plt.title("Patients per Cohort (Sex)")
plt.xlabel("Cohort")
plt.ylabel("Number of patients")
plt.xticks(rotation=0)
plt.legend(title="Sex")
plt.tight_layout()
plt.show()


In [ ]:
#Are the cohorts themselves different in age??
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_cohorts,
    x="CONCOHORT",
    y="age",
    color="orange"
)

plt.title("Age distribution by cohort (excluding cohort 3)")
plt.tight_layout()
plt.show()

In [ ]:
df_cohorts["subgroup"].value_counts().plot(kind="bar", color = ['lightblue'])
plt.title("Subgroup sizes")
plt.ylabel("Number of subjects")
plt.tight_layout()
plt.show()


In [ ]:
#sex/gender balance per subgroup
sex_map = {0: "Female", 1: "Male"}
df_plot = df_cohorts.copy()
df_plot["SEX_lbl"] = df_plot["SEX"].map(sex_map)


ct = (
    df_plot
    .groupby(["subgroup", "SEX_lbl"])
    .size()
    .unstack(fill_value=0)
)

ct.plot(kind="bar", stacked=True, color = ["pink", "lightblue"])
plt.title("Sex distribution across subgroups")
plt.legend(title="Sex")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Subgroup")
plt.ylabel("Number of Subjects")
plt.tight_layout()
plt.show()


In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal"}
df_cohorts = df_cohorts.copy()
df_cohorts["CONCOHORT_LABEL"] = df_cohorts["CONCOHORT"].map(cohort_map)

subgroups_counts = df_cohorts["subgroup"].value_counts()
valid_subgroups = subgroups_counts[subgroups_counts >= 20].index
df_filtered = df_cohorts[df_cohorts["subgroup"].isin(valid_subgroups)]

filtered_counts = df_filtered["subgroup"].value_counts()

order = ["Healthy Control"] + [
    sg for sg in filtered_counts.index
    if sg != "Healthy Control"
]

plt. figure(figsize=(8, 6))

sns.countplot(
    data=df_filtered,
    x="subgroup",
    hue="CONCOHORT_LABEL",
    palette="Set3",
    order=order
)

plt.title("Subgroup sizes by cohort (n ≥ 20, Concohort 3 excluded)")
plt.ylabel("Number of subjects")
plt.xlabel("Subgroup")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Concohort", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

pd.crosstab(df_filtered["subgroup"], df_filtered["CONCOHORT_LABEL"])

In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal"}
df_cohorts = df_cohorts.copy()
df_cohorts["CONCOHORT_LABEL"] = df_cohorts["CONCOHORT"].map(cohort_map)

subgroups_counts = df_cohorts["subgroup"].value_counts()
valid_subgroups = subgroups_counts[subgroups_counts >= 20].index
df_filtered = df_cohorts[df_cohorts["subgroup"].isin(valid_subgroups)]

filtered_counts = df_filtered["subgroup"].value_counts()

order = ["Healthy Control"] + [
    sg for sg in filtered_counts.index
    if sg != "Healthy Control"
]


plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df_filtered,
    x="subgroup",
    y="age",
    hue="CONCOHORT_LABEL",
    hue_order=["HC", "PD", "Prodromal"],
    palette="Set3",
    order=order,
    width=0.6
)

plt.title("Age distribution by subgroup and cohort (n ≥ 20, Concohort 3 excluded)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Concohort", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

df_filtered.groupby(["subgroup", "CONCOHORT"]).size().unstack(fill_value=0)

In [ ]:
final_df.shape

In [ ]:
df_cohorts.shape

In [ ]:
from matplotlib.lines import Line2D

df = df_cohorts.copy()

# --- cohort labels ---
cohort_map = {1: "PD", 4: "Prodromal PD", 2: "HC"}
order = ["PD", "Prodromal PD", "HC"]

# keep only cohorts we want (safe even if already filtered)
df = df[df["CONCOHORT"].isin(cohort_map.keys())].copy()
df["CONCOHORT_lbl"] = df["CONCOHORT"].map(cohort_map)

# --- numeric age ---
df["age"] = pd.to_numeric(
    df["age"].astype("string").str.strip().str.replace(",", ".", regex=False),
    errors="coerce"
)

# --- color by subgroup (everything else black) ---
color_map = {
    "Hyposmia": "tab:blue",
    "RBD": "tab:orange",
    "LRRK2": "tab:green",
    "GBA": "tab:red",
}

df["pt_color"] = (
    df["subgroup"]
    .astype("string")
    .str.strip()
    .map(color_map)
    .fillna("black")
)

# --- outliers per cohort via 1.5*IQR ---
df["is_outlier"] = False
bounds = {}

for lbl in order:
    s = df.loc[df["CONCOHORT_lbl"].eq(lbl), "age"].dropna()
    if len(s) == 0:
        continue

    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    bounds[lbl] = (lower, upper)

    df.loc[
        df["CONCOHORT_lbl"].eq(lbl) &
        ((df["age"] < lower) | (df["age"] > upper)),
        "is_outlier"
    ] = True

print("Outlier counts by cohort:")
print(
    df[df["is_outlier"]]
    .groupby("CONCOHORT_lbl")
    .size()
    .reindex(order)
    .fillna(0)
    .astype(int)
)

# --- plot ---
plt.figure(figsize=(12, 5))

# boxplots (no default fliers)
data = [
    df.loc[df["CONCOHORT_lbl"].eq(lbl), "age"].dropna().values
    for lbl in order
]
plt.boxplot(data, tick_labels=order, showfliers=False)

# overlay points
for i, lbl in enumerate(order, start=1):
    sub = df[df["CONCOHORT_lbl"].eq(lbl)].copy()
    x = np.random.normal(loc=i, scale=0.06, size=len(sub))

    # non-outliers
    non_out = ~sub["is_outlier"].to_numpy()
    plt.scatter(
        x[non_out],
        sub.loc[~sub["is_outlier"], "age"],
        color="gray", alpha=0.25, s=18
    )

    # outliers
    out_sub = sub[sub["is_outlier"]]
    plt.scatter(
        x[sub["is_outlier"].to_numpy()],
        out_sub["age"],
        c=out_sub["pt_color"].values,
        s=60, edgecolors="k", linewidths=0.5
    )

# legend
handles = [
    Line2D([0],[0], marker='o', color='w', label=k,
           markerfacecolor=v, markeredgecolor='k', markersize=8)
    for k, v in color_map.items()
]

handles.append(
    Line2D([0],[0], marker='o', color='w', label="Other",
           markerfacecolor="black", markeredgecolor='k', markersize=8)
)

plt.legend(
    handles=handles,
    title="subgroup (outliers only)",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.title("Age distribution by cohort (outliers highlighted by subgroup)")
plt.xlabel("Cohort")
plt.ylabel("Age (years)")
plt.tight_layout()
plt.show()



In [ ]:
# Make sure this runs AFTER the outlier computation cell

cols = ["PATNO", "age", "CONCOHORT_lbl", "subgroup"]

out_table = (
    df.loc[df["is_outlier"], cols]
      .drop_duplicates()  # in case PATNO appears more than once
      .sort_values(["CONCOHORT_lbl", "age"])
)

print("Outlier patients (PATNO) by cohort:")
print(out_table)

# Optional: show counts per cohort (safe even if some cohorts absent)
order = ["PD", "Prodromal PD", "HC"]

print("\nOutlier counts per cohort:")
print(
    out_table["CONCOHORT_lbl"]
        .value_counts()
        .reindex(order)
        .fillna(0)
        .astype(int)
)


- want to remove all outliers for HC, all outlier of Sporadic PD

In [ ]:
## Removing unwanted outliers all healthy controls and Sporadic PD outlier case
df = df[
    ~(
        (df["is_outlier"] & (df["subgroup"] == "Sporadic PD")) |
        (df["is_outlier"] & (df["CONCOHORT_lbl"] == "HC"))
    )
].copy()

print("Remaining outliers:")
print(df[df["is_outlier"]]["CONCOHORT_lbl"].value_counts())

In [ ]:
df.shape


In [ ]:
df.head()

In [ ]:
df = df.drop(columns=["CONCOHORT_LABEL", "CONCOHORT_lbl", "pt_color", "is_outlier"])
df.shape

In [ ]:
out_path = "df.csv"

df.to_csv(
    out_path,
    index=False,        # don’t write pandas index as a column
    encoding="utf-8",   # safe default
    na_rep=""           # empty cells instead of "NaN"
)

print(f"Saved {len(df)} rows to {out_path}")

### Checking which individuals have LEDD values --> removing them in both PD patients and Prodromal cases

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df = df[df["CONCOHORT"].isin(cohort_map)].copy()
df["CONCOHORT_lbl"] = df["CONCOHORT"].map(cohort_map)
df["LEDD"] = pd.to_numeric(df["LEDD"].astype("string").str.replace(",", ".", regex=False), errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# A) fraction LEDD == 0 vs > 0
tmp = df.assign(on_meds=df["LEDD"].fillna(0).gt(0))
ct = (tmp.groupby("CONCOHORT_lbl")["on_meds"]
        .mean()
        .reindex(order))

axes[0].bar(ct.index, ct.values)
axes[0].set_title("Percentage with nonzero LEDD scores")
axes[0].set_ylabel("Percentage")
axes[0].set_ylim(0, 1)

# B) distribution among those with LEDD > 0
df_nz = df[df["LEDD"] > 0].copy()
data = [df_nz.loc[df_nz["CONCOHORT_lbl"].eq(lbl), "LEDD"].dropna() for lbl in order]

axes[1].boxplot(data, tick_labels=order, showfliers=True)
axes[1].set_title("LEDD among non-zero only")
axes[1].set_ylabel("LEDD")

plt.tight_layout()
plt.show()

In [ ]:
# ensure LEDD numeric
df["LEDD"] = pd.to_numeric(df["LEDD"].astype(str).str.replace(",", "."), errors="coerce")

# PATNOs to drop (any record with LEDD > 0)
ids_to_drop = df.loc[df["LEDD"].gt(0), "PATNO"].dropna().unique()

# drop them
df = df.loc[~df["PATNO"].isin(ids_to_drop)].copy()

print(f"Dropped IDs: {len(ids_to_drop)}")
print(f"Any of those IDs still present?: {df['PATNO'].isin(ids_to_drop).sum()}")

In [ ]:
nonzero_ledd = (
    df.loc[df["LEDD"].gt(0), ["PATNO", "LEDD", "subgroup"]]
      .drop_duplicates()
      .sort_values("LEDD", ascending=False)
)
display(nonzero_ledd)
print("Individuals with LEDD > 0:", nonzero_ledd["PATNO"].nunique())
df.shape

## Saving dataframe "df" as csv file: Includes all Prodromal cases (including genetic variants), PD, and HC
## Saving a new dataframe "df1" as csv file: Includes HC, Sporadic PD (including PD with RBD)(excluding genetic variants), and prodromal PD (hyposmia + RBD)(excluding genetic variants) - after ledd removed

In [ ]:
## keeping only the wanted subgroups

df["subgroup"] = df["subgroup"].astype("string").str.strip()

keep = (
    (df["CONCOHORT"].eq(2)) |  #HC
    (df["CONCOHORT"].eq(1) & df["subgroup"].isin(["Sporadic PD", "RBD"])) |  #PD, note - no RBD left after removing sporadic PD missing values for "agediag"
    (df["CONCOHORT"].eq(4) & df["subgroup"].isin(["Hyposmia", "RBD"]))  #prodromal 
)

df1= df[keep].copy()

print("Rows kept:", len(df1))
print("Unique PATNO kept:", df1["PATNO"].nunique())

print("\nCounts by cohort:")
print(df1["CONCOHORT"].value_counts().sort_index())

print("\nCounts by cohort + subgroup:")
print(df1.groupby(["CONCOHORT", "subgroup"]).size().sort_index())

In [ ]:
df1.head(5)

In [ ]:
df1=df1.drop(columns=["CONCOHORT_lbl"])
df1.shape

In [ ]:
out_path = "df1.csv"

df1.to_csv(
    out_path,
    index=False,        # don’t write pandas index as a column
    encoding="utf-8",   # safe default
    na_rep=""           # empty cells instead of "NaN"
)

print(f"Saved {len(df1)} rows to {out_path}")

### after checking LEDD cases for PD patients we noticed that the ones with LEDD values are genetic variants of diagnosed PD - there were only 7 that where both LEDD and Sporadic PD, everyone else where removed when creating the wanted subgroup dataset